In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False 
import pickle
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from itertools import combinations_with_replacement
from sklearn.preprocessing import StandardScaler
cm = 1/2.54
# 字号
fontsize = 8
seed = 2025

In [51]:
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\data\HI.pkl', 'rb') as file:
    df = pickle.load(file)
# 删除含有NAN的行
HI = df.dropna()

In [52]:
HI.columns

Index(['CC_Var', 'CC_Ske', 'CC_Max', 'CC_Min', 'CC_Mean', 'CC_Kur', 'CV_Var',
       'CV_Ske', 'CV_Max', 'CV_Min', 'CV_Mean', 'CV_Kur', 'RE_Var', 'RE_Ske',
       'RE_Max', 'RE_Min', 'RE_Mean', 'RE_Kur', 'cycle', 'capacity', 'id'],
      dtype='object')

In [53]:
# 读取相关性系数,csv文件
cor = pd.read_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\3-HI\皮尔逊相关系数.csv',index_col=0)
cor_s = pd.read_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\3-HI\斯皮尔曼相关系数.csv',index_col=0)
index = cor.index

In [54]:
# cor和cor_s归一化至0-1，使用归一化器
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))
cor = scaler.fit_transform(cor)
cor_s = scaler.fit_transform(cor_s)
cor = pd.DataFrame(cor, index=index)
cor_s = pd.DataFrame(cor_s, index=index)

In [55]:
# cor和cor_s按index相加，选取各阶段归一化后皮尔逊相关系数和斯皮尔曼相关系之和最大的前3个特征
cor_sum = cor + cor_s
A = ['CC', 'CV', 'RE']
B = ['Kur', 'Max', 'Mean', 'Min', 'Ske', 'Var']
desired_order = [f"{a}_{b}" for a in A for b in B]
final_order = ['capacity'] + desired_order
# 3. 核心步骤：使用 reindex 按指定的列表顺序重新排列 Series 的行
cor = cor.reindex(final_order)
cor_s = cor_s.reindex(final_order)
cor_sum.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\归一化后相关系数和.csv')

CC选择CC_Kur CC_Min	CC_Mean  
CV选择CV_Kur CV_Ske CV_Mean  
RE选择RE_Mean RE_Max RE_Min

In [56]:
# 生成各阶段的数据集
data_CC = HI[['CC_Kur', 'CC_Min', 'CC_Mean', 'cycle', 'capacity', 'id']]
data_CV = HI[['CV_Kur', 'CV_Ske', 'CV_Mean', 'cycle', 'capacity', 'id']]
data_RE = HI[['RE_Mean', 'RE_Max', 'RE_Min', 'cycle', 'capacity', 'id']]
# 保存到G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\HI_CC.pkl
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\HI_CC.pkl', 'wb') as file:
    pickle.dump(data_CC, file)
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\HI_CV.pkl', 'wb') as file:
    pickle.dump(data_CV, file)
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\HI_RE.pkl', 'wb') as file:
    pickle.dump(data_RE, file)

# 按id分为训练、验证和测试集，比例为6:2:2
id = data_CC['id'].unique()
np.random.seed(seed)
np.random.shuffle(id)
train_id = id[:int(len(id)*0.6)]
val_id = id[int(len(id)*0.6):int(len(id)*0.8)]
test_id = id[int(len(id)*0.8):]
# 生成训练、验证和测试集
def generate_dataset(data, path):
    train = data[data['id'].isin(train_id)]
    val = data[data['id'].isin(val_id)]
    test = data[data['id'].isin(test_id)]
    with open(path, 'wb') as file:
        pickle.dump([train, val, test], file)
generate_dataset(data_CC, r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\dataset_HI_CC.pkl')
generate_dataset(data_CV, r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\dataset_HI_CV.pkl')
generate_dataset(data_RE, r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\4-SOH_Estimation\dataset_HI_RE.pkl')